# IMPLEMENTATION OF GAUSS ELIMINATION ALGORITHM, LU FACTORIZATION AND GAUSS-JORDAN ELIMINATION

We know the characteristics of all these algorithms, our goal here is to experiment their numerical implementation in python.

## GUASS ELIMATION.

Let's start from a review of all the possible useful operation with arrays in numpy (regarding entries selection and overwriting, but also reshaping).

In [1]:
import numpy as np

A = np.array([
    [1,3,6],
    [4,3,8],
    [5,9,0],
    ], dtype = float)

print(A.shape) # A dimensions

print(A[0,1])   # Element of A in position 0,1 which is first row second column, pos i,j

print(A[0,:])   # Select the first row, equivalently A[0]

print(A[:,1])  # Select the second column, a syntex similar to that above is not possible in the case of columns

print(A[:2,:])  # First two rows, all the columns

print(A[:, :2]) # All rows first two columns

print(A[1:, 2:])    # Rows from index one onward, columns from index 2 onward

# So the general syntax is [i:n, j:m] where i is the first index of the rows of interest that we want to extract, while n-1 is the index of the last
# row (so the n-th row is actually excluded)
# Simingly for the columns. So this gives us the possibility to extract blocks from a given matrix basically

print('\n', A[0:2, 0:2])  #Upper left 2x2 block of A, it is equivalent to say: "exctract all the element in the first 2 rows and first two columns"

B = np.array([
    [1, 2.4, -4, 5.5],
    [6, 8, 0, 28],
    [7.7, 11, 17, 39],
    [25, 9, 49, 6.89]
], dtype = float)

print('\n', B[1:3, 1:3])    # Central 2x2 block of the matrix

# Leave the second index of i:n unspecified mean take all the columns/rows from the i-th one, leaving the fisrt unspecified means take all the 
# rows/columns up to the n-1.
# That's all about entries exctraction

print('\n', B[1:3, 0:3])




(3, 3)
3.0
[1. 3. 6.]
[3. 3. 9.]
[[1. 3. 6.]
 [4. 3. 8.]]
[[1. 3.]
 [4. 3.]
 [5. 9.]]
[[8.]
 [0.]]

 [[1. 3.]
 [4. 3.]]

 [[ 8.  0.]
 [11. 17.]]

 [[ 6.   8.   0. ]
 [ 7.7 11.  17. ]]


In [2]:
# Now some more advance entries extraction commands

print(B[::2, :])    # Extract all the rows jumping from 0-th to 2-nd to 4-th etc... basically skipping one index each time, starting from 0

print('\n', B[1::2, :]) # Here we start from the 1-st row, which means print all the odd-indiced rows, the command above exctrats the even-indeced ones

print('\n', B[:, ::3])  # The same can be done with the columns too, obviously the specified "jump" it is not strickly 2, it can be any int numbers

print('\n', B[:, ::4])  # If we goes out of bounds (B has j = 0,1,2,3) the command prints only the possible entries, in this case only the first column

# In this way it is obviously possible two exctract any set of entries, ordered as you wish, from the matrix

# if we want to exactly speficy rows an columns we can pass a list:

print('\n', B[[1,3], [0,2]])    # this extract the element in 1,0 and in 3,2

print('\n', B[:, [2,3]])    # It prints all the entries of columns 2 and 3

print('\n', B[[0,1], :])    # First two rows, that can be done also with B[:2, :]

# To select every intersection of rows i, n and columns j, m use np.ix_

print('\n', B[np.ix_([0,2], [1,3])])



[[ 1.   2.4 -4.   5.5]
 [ 7.7 11.  17.  39. ]]

 [[ 6.    8.    0.   28.  ]
 [25.    9.   49.    6.89]]

 [[ 1.    5.5 ]
 [ 6.   28.  ]
 [ 7.7  39.  ]
 [25.    6.89]]

 [[ 1. ]
 [ 6. ]
 [ 7.7]
 [25. ]]

 [ 6. 49.]

 [[-4.    5.5 ]
 [ 0.   28.  ]
 [17.   39.  ]
 [49.    6.89]]

 [[ 1.   2.4 -4.   5.5]
 [ 6.   8.   0.  28. ]]

 [[ 2.4  5.5]
 [11.  39. ]]


In [3]:
# Boolean masks

print(B[B > 1]) # To select individual element of B that respect a specific condition. Returns a 1d array

print('\n', B[(B>1) | (B%2 != 0)])  # You can do whatever you want, but REMEMBER THE PARENTESIS (instead of 'or' use | and & for 'and', the bitwise
                                    # logical operation basically)

# Then logical masks on specific rows and columns

print('\n', B[B[:, 3]>25, :])    # Rows whose third entries exceed 30

print('\n', B[:, B[0,:]<=1])    # Columns whose first entries are at lest 1

#NOTE: After selecting a specific set of elements of an array, is it possible to overwrite them: B[:, B[0,:]<=1] = 0, put to zero all the selected elements
# A[:,1] = 0 set the selected column to zero.

[ 2.4   5.5   6.    8.   28.    7.7  11.   17.   39.   25.    9.   49.
  6.89]

 [ 1.    2.4   5.5   6.    8.   28.    7.7  11.   17.   39.   25.    9.
 49.    6.89]

 [[ 6.   8.   0.  28. ]
 [ 7.7 11.  17.  39. ]]

 [[ 1.  -4. ]
 [ 6.   0. ]
 [ 7.7 17. ]
 [25.  49. ]]


In [4]:
# Keeping or removing an axis

print(B[1, :].shape)    # Gives the shape of the array obtain exctracing the second row (1d array of 4 entries)

print('\n', B[1:3, :].shape)    # Gives the shape of the array composed by the second and the third rows of B (2x4 matrix)

print('\n', B[:, 2:3].shape)    # Same thing can be done with columns

# An alternative idea of this operation is that give an interger index is equivalent to remove an axis, while giving a slice (n:m) keeps that axis.
# But it is kinda tautologic


(4,)

 (2, 4)

 (4, 1)


In [5]:
# Tecnical detail: ordinary slices share data with the original array, so to have an indipend quantity (like extracting an array from a bigger one and
# operate on it independently from the original one) we use:

row = A[0,:]
row = A[0,:].copy() # Acompletely independent object from the row in the line above

In [6]:
# Useful operation for gauss elimination:

A[1,:] -= 5.*A[0,:] # which multiply by a factor 5 the first row and subtrac it to the second one

In [7]:
# Reshape:

print(A.reshape(1,3,3)) # It change the dimensions of an array. You can do it if the total number of elements needed to fill the new array is
                        # equal to the number of elements in the original array

# nice trick
print('\n', A.reshape(-1))  # Reduce the number of dimensions of -1

A.reshape(-1).copy()

print('\n', A.reshape(-1, 1))   # (9,1) column 

print('\n', A.reshape(1, -1))   # (1,9) row

# usefull for transpose 1d array in lin. sys. when needed. It DOES NOT TRANSPOSE MATRCES

[[[  1.   3.   6.]
  [ -1. -12. -22.]
  [  5.   9.   0.]]]

 [  1.   3.   6.  -1. -12. -22.   5.   9.   0.]

 [[  1.]
 [  3.]
 [  6.]
 [ -1.]
 [-12.]
 [-22.]
 [  5.]
 [  9.]
 [  0.]]

 [[  1.   3.   6.  -1. -12. -22.   5.   9.   0.]]


## GUASS ELIMANTION.

Now we can actually start experimenting with proper Gauss elimination

In [8]:
import numpy as np

A = np.array([
    [ 2,  3,  1,  5,  4],
    [ 7,  1,  4,  2,  6],
    [-9,  2,  5,  1,  3],
    [ 4, -8,  2,  6,  1],
    [ 1,  5, -7,  3,  9]
], dtype=float)

print(A)
print(A[:,0].shape[0])

B = A.copy()

for i in range(A[:,0].shape[0]-1):
    # Let's start ordering the columns
    where_max = i + np.argmax(abs(A[i:, i]))
    row = A[i, :].copy()
    A[i,:] = A[where_max, :]
    A[where_max, :] = row
    # Now the rows should have been exchange one with other

G = A.copy()

print('\n',A)



[[ 2.  3.  1.  5.  4.]
 [ 7.  1.  4.  2.  6.]
 [-9.  2.  5.  1.  3.]
 [ 4. -8.  2.  6.  1.]
 [ 1.  5. -7.  3.  9.]]
5

 [[-9.  2.  5.  1.  3.]
 [ 4. -8.  2.  6.  1.]
 [ 1.  5. -7.  3.  9.]
 [ 2.  3.  1.  5.  4.]
 [ 7.  1.  4.  2.  6.]]


In [9]:
#even better for the row swap
print(B.shape, '\n', B.shape[0])

for i in range(B.shape[0]-1):
    # Let's start ordering the columns
    where_max = i + np.argmax(abs(B[i:, i]))
    B[[i, where_max], :] = B[[where_max, i], :]
    # Now the rows should have been exchange one with other

print('\n',B)

C = B.copy()

G.shape

J = G.copy()

J[1,1] = 0
J[2,3] = 0

print(G != B)

print('\n', np.argwhere(J != B))    # Returns the positions of the entries of G and B which differ between them

# This version correctly work


(5, 5) 
 5

 [[-9.  2.  5.  1.  3.]
 [ 4. -8.  2.  6.  1.]
 [ 1.  5. -7.  3.  9.]
 [ 2.  3.  1.  5.  4.]
 [ 7.  1.  4.  2.  6.]]
[[False False False False False]
 [False False False False False]
 [False False False False False]
 [False False False False False]
 [False False False False False]]

 [[1 1]
 [2 3]]


In [10]:
# Now the actual Gauss elimination

print(C)

for i in range(A.shape[0]):
    if C[i,i] == 0:
        C[[i, i+1], :] = C[[i+1, i], :]
        continue
    else:
        C[i,:] /= C[i,i]

    # print('\n', C[i,:], '\n', C)
    # print(i)
    # print('\n', range(i + 1, A.shape[0]))

    for j in range(i + 1, A.shape[0]):
        if C[j,i] == 0:
            continue
        else:
            C[j,:] -= C[j,i]*C[i,:]
            # print(C[j,:], '\n', C, '\n', j)

print('\n', C)

[[-9.  2.  5.  1.  3.]
 [ 4. -8.  2.  6.  1.]
 [ 1.  5. -7.  3.  9.]
 [ 2.  3.  1.  5.  4.]
 [ 7.  1.  4.  2.  6.]]

 [[ 1.         -0.22222222 -0.55555556 -0.11111111 -0.33333333]
 [-0.          1.         -0.59375    -0.90625    -0.328125  ]
 [-0.         -0.          1.         -2.34579439 -3.30373832]
 [ 0.          0.          0.          1.          1.07928719]
 [ 0.          0.          0.          0.          1.        ]]


In [11]:
C = np.array([
    [4, 4, 0, 0],
    [3, 3, 1, 0],
    [2, 2, 0, 1],
    [1, 0, 0, 0]
], dtype=float)

A = C.copy()   # Your loop uses A.shape

# Now the actual Gauss elimination

print(C)

for i in range(A.shape[0]-1):
    # Let's start ordering the columns
    where_max = i + np.argmax(abs(C[i:, i]))
    C[[i, where_max], :] = C[[where_max, i], :]
    # Now the rows should have been exchange one with other

print('\n',C)

for i in range(A.shape[0]):
    if C[i,i] == 0:
        C[[i, i+1], :] = C[[i+1, i], :]
        continue
    else:
        C[i,:] /= C[i,i]

    # print('\n', C[i,:], '\n', C)
    # print(i)
    # print('\n', range(i + 1, A.shape[0]))

    for j in range(i + 1, A.shape[0]):
        if C[j,i] == 0:
            continue
        else:
            C[j,:] -= C[j,i]*C[i,:]
            # print(C[j,:], '\n', C, '\n', j)

print('\n', C)

# Pivot selection must happen inside the elimination loop: elimination changes
# the remaining entries, so choosing all pivots beforehand is insufficient.
# Before each elimination step, select the largest absolute entry in the
# current column among the remaining rows and swap it into the pivot position.
# Swapping only with the next row may leave a zero pivot; using continue
# after the swap also skips normalisation and elimination for that pivot.

[[4. 4. 0. 0.]
 [3. 3. 1. 0.]
 [2. 2. 0. 1.]
 [1. 0. 0. 0.]]

 [[4. 4. 0. 0.]
 [3. 3. 1. 0.]
 [2. 2. 0. 1.]
 [1. 0. 0. 0.]]


IndexError: index 4 is out of bounds for axis 0 with size 4

In [7]:
import numpy as np

C  = np.array([
    [ 2, -3,  5],
    [ 4,  1, -2],
    [-8,  2,  3],
    [ 1,  7,  4],
    [-3,  5, -6],
], dtype=float)

A = C.copy()   # Your loop uses A.shape

print(C)

for i in range(C.shape[0]):
    # Let's start ordering the columns
    # if C.shape[0] <= C.shape[1]:
    where_max = i + np.argmax(abs(C[i:, i]))

    if C[where_max, i] == 0:
        raise ValueError("Singular matrix: no nonzero pivot available.")

    C[[i, where_max], :] = C[[where_max, i], :]

    # Now we can proceed with the elimination
    C[i,:] /= C[i,i]

    for j in range(i + 1, C.shape[0]):
        if C[j,i] == 0:
            continue
        else:
            C[j,:] -= C[j,i]*C[i,:]
    
    print('\n', C)

    # This version of the Gauss-elimination algorithm works only for mtracies such that the number of the rows is smaller or equal to the number of
    # their columns. Otherwhise the line 'where_max = i + np.argmax(abs(C[i:, i]))' will raise an error.

    # Then it is clear that we cannot disregard the number of columns when we implement this algoritmh, which is a constrain in Gauss-elimination.
    # This is explain exactly from what the bug above underlines: at most I can have a number of pivot equal to the number of columns.

    # Now this statement is trivial is this is true for rows too, The number of pivots has to stay in between 0 and min(rows, columns). Then I have
    # to consider both when I perform the iteration: I can't indefinitly interate over the rows forgetting about the columns and vice versa for the 
    # iteration over the columns.

    # Well, actually there is one more fact to consider: the matrix might have some rows or columns linerally dependent. Which implication has this fact?
    # When I peform the iteration tring to order the rows I might find the entries below a certain row in a fixed column to be zeros: image a matrix
    # m x n where two pivots has already been found, let's say on row 1 and 2, there is the possibility to move to row 3 and find out that all the entries
    # from row 3 included all the way down to the last column are zeros. This means that on this column I won't have a pivot and this forces me to 
    # move to the next columns. This example takes into account matrix which can be reduced in the form REEF, but that do not have pivots on each column.

[[ 2. -3.  5.]
 [ 4.  1. -2.]
 [-8.  2.  3.]
 [ 1.  7.  4.]
 [-3.  5. -6.]]

 [[ 1.    -0.25  -0.375]
 [ 0.     2.    -0.5  ]
 [ 0.    -2.5    5.75 ]
 [ 0.     7.25   4.375]
 [ 0.     4.25  -7.125]]

 [[ 1.         -0.25       -0.375     ]
 [ 0.          1.          0.60344828]
 [ 0.          0.          7.25862069]
 [ 0.          0.         -1.70689655]
 [ 0.          0.         -9.68965517]]

 [[ 1.         -0.25       -0.375     ]
 [ 0.          1.          0.60344828]
 [-0.         -0.          1.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]]


IndexError: index 3 is out of bounds for axis 1 with size 3

In [10]:
# Then, how can I implement the elimination algorithm in a way that it works also for matrices which enter in the case above?

# I need to eventually skip to the next column if the current one has only null entries. This means that I need to iterate on the columns, remembering
# in which row I am.

import numpy as np

A = np.array([
    [4, 8, 0, 4, 0, 0],
    [2, 4, 0, 0, 2, 0],
    [0, 0, 0, 4, 2, 0],
    [0, 0, 0, 0, 0, 2],
], dtype=float)

B = A.copy()

pivot_row  = 0  # To keep track of the row in which I am

pivot_cols = []

for col in range(B.shape[1]):

    if pivot_row == B.shape[0]: # Obviously we must not continue iterating once the last has been reached, so we exist from the loop.
        break

    where_max = pivot_row + np.argmax(abs(B[pivot_row:, col]))  # This command is the same of the algorithm above, given the row (which I initialize
                                                                # to zero) and the column in which I am (I'm iterating over the column) I found the 
                                                                # maximum entry in that column considering only those in the rows from the row in which
                                                                # I am to the last of the matrix.

    # What happens if my column has only zeros entries (in the rows I am considering)?

    if B[where_max, col] == 0:
        continue    # Stay on the same row (I don't change pivot_row) and skip to the next column
    else:
        B[[pivot_row, where_max], :] = B[[where_max, pivot_row], :] # Else proceed with the swapping

    # Now we can proceed with the actual elimination elimination
    B[pivot_row,:] /= B[pivot_row,col]  # Note also that now this operation never returns an error since with the control above I have already
                                        # checked that B[pivot_row,col] != 0

    for j in range(pivot_row + 1, B.shape[0]):
        if B[j,col] == 0:
            continue
        else:
            B[j,:] -= B[j,col]*B[pivot_row,:]
    
    pivot_row += 1  # Finally I increment the pivot_row by one to move to the next row

    pivot_cols.append(col)

    print('\n', B)

print(pivot_cols, len(pivot_cols))

# In this way I iterate both on columns and rows, but without an explicit nested for loop and I can also eventually skip the column if necessary
# Moreover it works for any matrix, independetly of its shape.


 [[ 1.  2.  0.  1.  0.  0.]
 [ 0.  0.  0. -2.  2.  0.]
 [ 0.  0.  0.  4.  2.  0.]
 [ 0.  0.  0.  0.  0.  2.]]

 [[1.  2.  0.  1.  0.  0. ]
 [0.  0.  0.  1.  0.5 0. ]
 [0.  0.  0.  0.  3.  0. ]
 [0.  0.  0.  0.  0.  2. ]]

 [[1.  2.  0.  1.  0.  0. ]
 [0.  0.  0.  1.  0.5 0. ]
 [0.  0.  0.  0.  1.  0. ]
 [0.  0.  0.  0.  0.  2. ]]

 [[1.  2.  0.  1.  0.  0. ]
 [0.  0.  0.  1.  0.5 0. ]
 [0.  0.  0.  0.  1.  0. ]
 [0.  0.  0.  0.  0.  1. ]]
[0, 3, 4, 5] 4


## LU FACTORIZATION

Next step is to implement the LU factorization (before the Gauss-Jordan elimination, since this algorithm is just a natural extenction of the Gauss elimination and LU factorizaztion).

## From row operations to LU factorisation

At pivot step $k$, Gaussian elimination clears the entries below the
pivot by subtracting a multiple of the pivot row. Let $C$ be the
current working matrix. For each row $i>k$, define

$$
\ell_{ik}=\frac{C_{ik}}{C_{kk}},
\qquad
R_i \leftarrow R_i-\ell_{ik}R_k.
$$

The pivot row stays unnormalised, so the division appears in the multiplier.

### The matrix for one row operation

Let $e_i$ be the $i$-th standard basis vector. Left multiplication by
$e_i e_k^{\mathsf T}$ places a copy of row $k$ in row $i$, with zeros
everywhere else. Therefore, the elimination matrix and its inverse are

$$
E_{ik}=I-\ell_{ik}e_i e_k^{\mathsf T},
\qquad
E_{ik}^{-1}=I+\ell_{ik}e_i e_k^{\mathsf T}.
$$

$E_{ik}$ subtracts $\ell_{ik}$ times row $k$ from row $i$.
Its inverse adds the same multiple back. For example, eliminating
entry $(3,1)$ in a matrix with three rows gives

$$
E_{31}=
\begin{pmatrix}
1&0&0\\
0&1&0\\
-\ell_{31}&0&1
\end{pmatrix},
\qquad
E_{31}^{-1}=
\begin{pmatrix}
1&0&0\\
0&1&0\\
\ell_{31}&0&1
\end{pmatrix}.
$$

### Building $L$ from the multipliers

Suppose no row swaps are needed. For a $3\times3$ matrix, eliminate
entries $(2,1)$ and $(3,1)$, then $(3,2)$. The operations give

$$
U=E_{32}E_{31}E_{21}A.
$$

Undoing them gives

$$
A=\underbrace{E_{21}^{-1}E_{31}^{-1}E_{32}^{-1}}_{L}\,U,
\qquad
L=
\begin{pmatrix}
1&0&0\\
\ell_{21}&1&0\\
\ell_{31}&\ell_{32}&1
\end{pmatrix}.
$$

The multipliers appear directly below the diagonal of $L$. In code,
when eliminating entry $C_{ik}$, store its multiplier in $L_{ik}$.
The working matrix becomes $U$ as elimination proceeds.

### Row swaps and rectangular matrices

With partial pivoting, a row swap is represented by an identity matrix
with the corresponding two rows exchanged. The combined row permutation
is represented by $P$, and the factorisation satisfies $PA=LU$.

If rows are swapped at step $k$, swap those rows in the working matrix
and in the permutation vector. Also swap their entries in the columns
of $L$ already filled by earlier steps, `L[:, :k]`. Those multipliers
must follow the rows they describe.

Each elementary row operation acts on the rows, so its matrix is
$m\times m$ even when $A$ is $m\times n$. For the compact factorisation,
with $r=\min(m,n)$, the final factors have shapes $L\in\mathbb R^{m\times r}$
and $U\in\mathbb R^{r\times n}$.

In [ ]:
import numpy as np

A = np.array([
    [0, 2, 4, 0,  8,  2],
    [0, 4, 8, 4,  0, 12],
    [0, 0, 0, 8, -4, 16],
], dtype=float)

U = A.copy()

m = A.shape[0]

L = np.eye(m , dtype = float)   # This command actually creates an mxm identity matrix
P = np.eye(m, dtype = float)    # P is the square matrix used to keep track of the rows swappings

pivot_row  = 0  # To keep track of the row in which I am

pivot_cols = []

for col in range(U.shape[1]):

    # To save the single operations of the elimination algorithm.
    L_i = np.eye(m, dtype = float)

    if pivot_row == U.shape[0]: # Obviously we must not continue iterating once the last has been reached, so we exist from the loop.
        break

    where_max = pivot_row + np.argmax(abs(U[pivot_row:, col]))

    # What happens if my column has only zeros entries (in the rows I am considering)?

    if U[where_max, col] == 0:
        continue    # Stay on the same row (I don't change pivot_row) and skip to the next column
    else:
        U[[pivot_row, where_max], :] = U[[where_max, pivot_row], :] # Else proceed with the swapping
        # Saving the swapping of the rows
        P[[pivot_row, where_max], :] = P[[where_max, pivot_row], :]
        L[[pivot_row, where_max], :pivot_row] = L[[where_max, pivot_row], :pivot_row]

    # Now we can proceed with the actual elimination
    L_i[pivot_row,pivot_row] = U[pivot_row,col]    # Saving the reascaling of the row
    U[pivot_row,:] /= U[pivot_row,col]  # Note also that now this operation never returns an error since with the control above I have already
                                        # checked that B[pivot_row,col] != 0

    for j in range(pivot_row + 1, U.shape[0]):
        if U[j,col] == 0:
            continue
        else:
            L_i[j,pivot_row] = U[j,col]   # saving the row operation
            U[j,:] -= U[j,col]*U[pivot_row,:]

    pivot_row += 1  # Finally I increment the pivot_row by one to move to the next row

    pivot_cols.append(col)

    L = L @ L_i # Updating of the general L matrix

    print('\n', f"Iteration: {pivot_row}", '\n', U, '\n', L_i, '\n', L, '\n', P)

print(pivot_cols, len(pivot_cols))
print(np.allclose(P @ A, L @ U))



 Iteration: 1 
 [[ 0.  1.  2.  1.  0.  3.]
 [ 0.  0.  0. -2.  8. -4.]
 [ 0.  0.  0.  8. -4. 16.]] 
 [[4. 0. 0.]
 [2. 1. 0.]
 [0. 0. 1.]] 
 [[4. 0. 0.]
 [2. 1. 0.]
 [0. 0. 1.]] 
 [[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]

 Iteration: 2 
 [[ 0.   1.   2.   1.   0.   3. ]
 [ 0.   0.   0.   1.  -0.5  2. ]
 [ 0.   0.   0.   0.   7.   0. ]] 
 [[ 1.  0.  0.]
 [ 0.  8.  0.]
 [ 0. -2.  1.]] 
 [[ 4.  0.  0.]
 [ 0.  8.  0.]
 [ 2. -2.  1.]] 
 [[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]

 Iteration: 3 
 [[ 0.   1.   2.   1.   0.   3. ]
 [ 0.   0.   0.   1.  -0.5  2. ]
 [ 0.   0.   0.   0.   1.   0. ]] 
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 7.]] 
 [[ 4.  0.  0.]
 [ 0.  8.  0.]
 [ 2. -2.  7.]] 
 [[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
[1, 3, 4] 3
True


## GAUSS-JORDAN ELIMINATION.

Here we start from the LU decomposition above and we continue it eliminating the terms above the diagonal of U. Then filling accordingly L, we find the inverse of A

NOTE: A has to be square now.

In [23]:
import numpy as np
from elimination_algorithms import LU_decomposition

A = np.array([
    [ 2,  3,  1,  5,  4],
    [ 7,  1,  4,  2,  6],
    [-9,  2,  5,  1,  3],
    [ 4, -8,  2,  6,  1],
    [ 1,  5, -7,  3,  9]
], dtype=float)

L, U, P = LU_decomposition(A)

L1 = np.eye(U.shape[0], dtype=float)


print(U)

for i in range(U.shape[1]-1, 0, -1):
    L1_i = np.eye(U.shape[0], dtype=float)
    print(i)
    if U[i,i] == 0:
        continue
    else:
        for j in range(i-1, -1, -1):
            #print(j)
            L1_i[j,i] = U[j,i]
            U[j,:] -= U[j,i]*U[i,:]
          
    L1 = L1 @ L1_i

    print('\n', f"Iteration: {i}", '\n', U, '\n', L1_i, '\n', L1)

L = L @ L1

print(L, '\n', P @ A, '\n', U, '\n', np.allclose(L, P @ A))

[[ 1.         -0.22222222 -0.55555556 -0.11111111 -0.33333333]
 [-0.          1.         -0.59375    -0.90625    -0.328125  ]
 [ 0.          0.          1.          0.54152824  0.97508306]
 [ 0.          0.          0.          1.          1.48193393]
 [-0.         -0.         -0.         -0.          1.        ]]
4

 Iteration: 4 
 [[ 1.         -0.22222222 -0.55555556 -0.11111111  0.        ]
 [-0.          1.         -0.59375    -0.90625     0.        ]
 [ 0.          0.          1.          0.54152824  0.        ]
 [ 0.          0.          0.          1.          0.        ]
 [-0.         -0.         -0.         -0.          1.        ]] 
 [[ 1.          0.          0.          0.         -0.33333333]
 [ 0.          1.          0.          0.         -0.328125  ]
 [ 0.          0.          1.          0.          0.97508306]
 [ 0.          0.          0.          1.          1.48193393]
 [ 0.          0.          0.          0.          1.        ]] 
 [[ 1.          0.          0.

In [11]:
import numpy as np

A = np.array([
    [ 2,  3,  1,  5,  4],
    [ 7,  1,  4,  2,  6],
    [-9,  2,  5,  1,  3],
    [ 4, -8,  2,  6,  1],
    [ 1,  5, -7,  3,  9]
], dtype=float)

U = np.copy(A)

if U.shape[0] != U.shape[1]:
    raise ValueError("The given matrix is not square.")

m = U.shape[0]

E = np.eye(m , dtype = float)   # This command actually creates an mxm identity matrix

pivot_row = 0
pivot_cols = []

for col in range(U.shape[1]):

    E_i = np.eye(m, dtype = float)

    where_max = pivot_row + np.argmax(abs(U[pivot_row:, col]))

    if U[where_max, col] == 0:
        continue
    else:
        U[[pivot_row, where_max], :] = U[[where_max, pivot_row], :]
        E[[pivot_row, where_max], :] = E[[where_max, pivot_row], :]

    pivot = U[pivot_row, col]
    E_i[pivot_row, pivot_row] = 1/pivot
    U[pivot_row, :] /= U[pivot_row, col]

    for j in range(pivot_row + 1, U.shape[0]):
        if U[j,col] == 0:
            continue
        else:
            E_i[j,pivot_row] = -U[j,col]/pivot   # saving the row operation
            U[j,:] -= U[j,col]*U[pivot_row,:]

    pivot_row += 1  # Finally I increment the pivot_row by one to move to the next row

    pivot_cols.append(col)

    E = E_i @ E # Updating of the general E matrix

    print('\n', f"Iteration: {pivot_row}", '\n', f'U = \n{U}', '\n', f'E_i = \n{E_i}', '\n', f'E = \n{E}', '\n')

if len(pivot_cols) != m:
    raise np.linalg.LinAlgError("Matrix is singular: fewer than m pivots.")

for i in range(m-1, 0, -1):
    E_i = np.eye(m, dtype=float)

    for j in range(i-1, -1, -1):
        E_i[j,i] = -U[j,i]
        U[j,:] -= U[j,i]*U[i,:]

    E = E_i @ E

    print('\n', f"Iteration: {i}", '\n', f'U = \n{U}', '\n', f'E_i = \n{E_i}', '\n', f'E = \n{E}', '\n')

print(np.allclose(U, np.eye(m, dtype=float)))
print(np.allclose(E@A, U))

# import numpy as np


# U = A.copy()

# m = A.shape[0]

# L = np.eye(m , dtype = float)   # This command actually creates an mxm identity matrix
# P = np.eye(m, dtype = float)    # P is the square matrix used to keep track of the rows swappings

# pivot_row  = 0  # To keep track of the row in which I am

# pivot_cols = []

# for col in range(U.shape[1]):

#     # To save the single operations of the elimination algorithm.
#     L_i = np.eye(m, dtype = float)

#     if pivot_row == U.shape[0]: # Obviously we must not continue iterating once the last has been reached, so we exist from the loop.
#         break

#     where_max = pivot_row + np.argmax(abs(U[pivot_row:, col]))

#     # What happens if my column has only zeros entries (in the rows I am considering)?

#     if U[where_max, col] == 0:
#         continue    # Stay on the same row (I don't change pivot_row) and skip to the next column
#     else:
#         U[[pivot_row, where_max], :] = U[[where_max, pivot_row], :] # Else proceed with the swapping
#         # Saving the swapping of the rows
#         P[[pivot_row, where_max], :] = P[[where_max, pivot_row], :]
#         L[[pivot_row, where_max], :pivot_row] = L[[where_max, pivot_row], :pivot_row]

#     # Now we can proceed with the actual elimination
#     L_i[pivot_row,pivot_row] = U[pivot_row,col]    # Saving the reascaling of the row
#     U[pivot_row,:] /= U[pivot_row,col]  # Note also that now this operation never returns an error since with the control above I have already
#                                         # checked that B[pivot_row,col] != 0

#     for j in range(pivot_row + 1, U.shape[0]):
#         if U[j,col] == 0:
#             continue
#         else:
#             L_i[j,pivot_row] = U[j,col]   # saving the row operation
#             U[j,:] -= U[j,col]*U[pivot_row,:]

#     pivot_row += 1  # Finally I increment the pivot_row by one to move to the next row

#     pivot_cols.append(col)

#     L = L @ L_i # Updating of the general L matrix

#     print('\n', f"Iteration: {pivot_row}", '\n', U, '\n', L_i, '\n', L, '\n', P)

# print(pivot_cols, len(pivot_cols))
# print(np.allclose(P @ A, L @ U))



 Iteration: 1 
 U = 
[[ 1.         -0.22222222 -0.55555556 -0.11111111 -0.33333333]
 [ 0.          2.55555556  7.88888889  2.77777778  8.33333333]
 [ 0.          3.44444444  2.11111111  5.22222222  4.66666667]
 [ 0.         -7.11111111  4.22222222  6.44444444  2.33333333]
 [ 0.          5.22222222 -6.44444444  3.11111111  9.33333333]] 
 E_i = 
[[-0.11111111  0.          0.          0.          0.        ]
 [ 0.77777778  1.          0.          0.          0.        ]
 [ 0.22222222  0.          1.          0.          0.        ]
 [ 0.44444444  0.          0.          1.          0.        ]
 [ 0.11111111  0.          0.          0.          1.        ]] 
 E = 
[[ 0.          0.         -0.11111111  0.          0.        ]
 [ 0.          1.          0.77777778  0.          0.        ]
 [ 1.          0.          0.22222222  0.          0.        ]
 [ 0.          0.          0.44444444  1.          0.        ]
 [ 0.          0.          0.11111111  0.          1.        ]] 


 Iteration:

In [13]:
'''
TEST SCRIPT
'''

import numpy as np
from numpy.testing import assert_allclose, assert_array_equal, assert_raises

from elimination_algorithms import (
    Gauss_elimination,
    LU_decomposition,
    gauss_jordan_inverse,
)

RTOL = 1e-10
ATOL = 1e-12

# Each case gives the input, its expected echelon form, and the pivot columns.
elimination_cases = {
    "square with two successive row swaps": (
        [[8, 0, 0, 0],
         [2, 1, 1, 4],
         [4, 8, 0, 0],
         [6, 2, 4, 0]],
        [[1, 0, 0, 0],
         [0, 1, 0, 0],
         [0, 0, 1, 0],
         [0, 0, 0, 1]],
        [0, 1, 2, 3],
    ),
    "wide with skipped columns and a trailing column": (
        [[0, 2, 4, 6],
         [0, 4, 8, 4]],
        [[0, 1, 2, 1],
         [0, 0, 0, 1]],
        [1, 3],
    ),
    "wide with all pivot rows used early": (
        [[2, 4, 6],
         [0, 2, -2]],
        [[1, 2, 3],
         [0, 1, -1]],
        [0, 1],
    ),
    "tall and rank deficient with a pivot tie": (
        [[0,  0,  0],
         [0, -4, -8],
         [0,  4,  8],
         [0,  2,  4]],
        [[0, 1, 2],
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
        [1],
    ),
    "zero matrix": (
        np.zeros((3, 5)),
        np.zeros((3, 5)),
        [],
    ),
}

for name, (data, expected_ref, expected_pivots) in elimination_cases.items():
    A = np.array(data, dtype=float)
    original = A.copy()
    m = A.shape[0]

    # Gaussian elimination
    ref, pivot_cols = Gauss_elimination(A)
    assert_array_equal(A, original, err_msg=f"{name}: Gauss modified A")
    assert_array_equal(pivot_cols, expected_pivots, err_msg=name)
    assert_allclose(ref, expected_ref, rtol=RTOL, atol=ATOL, err_msg=name)

    # LU factorisation with the convention P @ A = L @ U
    L, U, P = LU_decomposition(A)
    assert_array_equal(A, original, err_msg=f"{name}: LU modified A")

    assert L.shape == P.shape == (m, m), name
    assert U.shape == A.shape, name
    assert_allclose(U, expected_ref, rtol=RTOL, atol=ATOL, err_msg=name)
    assert_array_equal(L, np.tril(L), err_msg=f"{name}: L is not triangular")

    assert np.isin(P, [0, 1]).all(), name
    assert_array_equal(P.sum(axis=0), np.ones(m), err_msg=name)
    assert_array_equal(P.sum(axis=1), np.ones(m), err_msg=name)

    assert_allclose(P @ original, L @ U, rtol=RTOL, atol=ATOL, err_msg=name)
    print(f"PASS  Gauss and LU — {name}")

# Seed the generator so the random test is reproducible.
rng = np.random.default_rng(42)

inverse_cases = {
    "1×1": np.array([[-4.0]]),
    "zero leading entry requires a swap": np.array([[0., 2.], [4., 6.]]),
    "two successive row swaps": elimination_cases[
        "square with two successive row swaps"
    ][0],
    "dense diagonally dominant matrix": rng.uniform(-1, 1, (8, 8))
        + 8 * np.eye(8),
}

for name, data in inverse_cases.items():
    A = np.array(data, dtype=float)
    original = A.copy()
    identity = np.eye(A.shape[0])

    inverse = gauss_jordan_inverse(A)

    assert_array_equal(A, original, err_msg=f"{name}: inverse modified A")
    assert_allclose(A @ inverse, identity, rtol=RTOL, atol=ATOL, err_msg=name)
    assert_allclose(inverse @ A, identity, rtol=RTOL, atol=ATOL, err_msg=name)
    assert_allclose(
        inverse, np.linalg.inv(A), rtol=RTOL, atol=ATOL, err_msg=name
    )
    print(f"PASS  Inverse — {name}")

# Singular inputs chosen to produce exact zero pivots in floating point.
for singular in (
    np.zeros((2, 2)),
    np.array([[1., 2.], [2., 4.]]),
    np.array([[0., 1.], [0., 2.]]),
):
    with assert_raises(np.linalg.LinAlgError):
        gauss_jordan_inverse(singular)

for invalid in (
    np.array([1., 2.]),
    np.ones((2, 3)),
    np.empty((0, 0)),
    np.array([[np.nan]]),
    np.array([[np.inf]]),
):
    with assert_raises(ValueError):
        gauss_jordan_inverse(invalid)

print("PASS  Inverse rejects singular and invalid inputs")
print("All tests passed.")


PASS  Gauss and LU — square with two successive row swaps
PASS  Gauss and LU — wide with skipped columns and a trailing column
PASS  Gauss and LU — wide with all pivot rows used early
PASS  Gauss and LU — tall and rank deficient with a pivot tie
PASS  Gauss and LU — zero matrix
PASS  Inverse — 1×1
PASS  Inverse — zero leading entry requires a swap
PASS  Inverse — two successive row swaps
PASS  Inverse — dense diagonally dominant matrix
PASS  Inverse rejects singular and invalid inputs
All tests passed.
